In [3]:
import cv2
import csv
import os
import math
import torch
import numpy as np
from collections import defaultdict
from ultralytics import YOLO

# CONFIGURACIÓN DE RUTAS Y DISPOSITIVO
VIDEO_ENTRADA = 'C0142.MP4'
VIDEO_SALIDA = 'resultado_C0142.mp4'
CSV_SALIDA = 'detecciones_seguimiento.csv'

MODELO_GENERAL_PATH = 'yolo11n.pt'
MODELO_MATRICULAS_PATH = 'best.pt'

# 0: person, 2: car, 3: motorcycle, 5: bus, 7: truck
CLASES_INTERES = [0, 2, 3, 5, 7]
CLASES_VEHICULOS = [2, 3, 5, 7]

DISPOSITIVO = 0 if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo de ejecución asignado: {DISPOSITIVO}")

# CARGA DE MODELOS
modelo_general = YOLO(MODELO_GENERAL_PATH)
modelo_matriculas = YOLO(MODELO_MATRICULAS_PATH)

# VÍDEO Y REGISTRO CSV
cap = cv2.VideoCapture(VIDEO_ENTRADA)
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el vídeo: {VIDEO_ENTRADA}")

ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
fps_salida = fps if fps > 0 else 25.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out = cv2.VideoWriter(
    VIDEO_SALIDA, 
    cv2.VideoWriter_fourcc(*'mp4v'), 
    fps_salida, 
    (ancho, alto)
)

f_csv = open(CSV_SALIDA, mode='w', newline='', encoding='utf-8')
csv_writer = csv.writer(f_csv)
csv_writer.writerow([
    'fotograma', 'tipo_objeto', 'confianza', 
    'identificador_tracking', 'x1', 'y1', 'x2', 'y2', 'matricula'
])

# CONFIGURACIÓN DEL SEGUIMIENTO Y CONTEO
# Evita el sobreconteo:
# - Filtra parpadeos y falsos positivos efímeros (mínimo de frames activo)
# - Filtra coches aparcados que parpadean tras oclusiones (mínimo desplazamiento)
# - Votación de clase por mayoría para que un ID no cuente en dos clases distintas
MIN_FRAMES_VALIDO = 10
MIN_DESPLAZAMIENTO = 40

# Estructura: {track_id: {'inicio': (cx, cy), 'actual': (cx, cy), 'frames': int, 'clases': defaultdict(int)}}
historial_tracks = defaultdict(lambda: {
    'inicio': None,
    'actual': None,
    'frames': 0,
    'clases': defaultdict(int)
})

# IDs únicos validados por clase dominante
conteo_unicos = {
    'person': set(),
    'car': set(),
    'motorcycle': set(),
    'bus': set(),
    'truck': set()
}

# Caché de matrículas: {track_id: {'rel_coords': (rx1, ry1, rx2, ry2), 'frame': frame_idx}}
cache_matriculas = {}

COLORES = {
    'person': (255, 105, 65),     # Azul/Cian
    'car': (40, 200, 40),         # Verde
    'motorcycle': (0, 165, 255),   # Naranja
    'bus': (200, 200, 0),         # Celeste
    'truck': (180, 0, 180),       # Magenta
    'matricula': (0, 255, 255)     # Amarillo
}

print(f"Procesando {total_frames} fotogramas...")

# BUCLE PRINCIPAL DE PROCESAMIENTO
frame_idx = 0

with torch.no_grad():
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_idx += 1

        # Seguimiento con ByteTrack y confianza mínima para estabilizar detecciones
        resultados_track = modelo_general.track(
            source=frame,
            persist=True,
            classes=CLASES_INTERES,
            tracker="bytetrack.yaml",
            conf=0.35,
            imgsz=640,
            device=DISPOSITIVO,
            verbose=False
        )

        if resultados_track and len(resultados_track) > 0 and resultados_track[0].boxes is not None:
            cajas = resultados_track[0].boxes

            for box in cajas:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                confianza = float(box.conf[0])
                clase_id = int(box.cls[0])
                nombre_clase = modelo_general.names[clase_id]
                track_id = int(box.id[0]) if box.id is not None else -1

                # Descartar ruido minúsculo en el horizonte (< 25px)
                w_box = x2 - x1
                h_box = y2 - y1
                if w_box < 25 or h_box < 25:
                    continue

                cx = int((x1 + x2) / 2)
                cy = int((y1 + y2) / 2)

                # Actualización del historial del objeto
                clase_dominante = nombre_clase
                if track_id != -1:
                    t = historial_tracks[track_id]
                    t['frames'] += 1
                    t['clases'][nombre_clase] += 1
                    if t['inicio'] is None:
                        t['inicio'] = (cx, cy)
                    t['actual'] = (cx, cy)

                    # Clase mayoritaria para evitar contar el mismo ID en dos categorías
                    clase_dominante = max(t['clases'].items(), key=lambda x: x[1])[0]

                    # Validación para el conteo:
                    # Persistencia mínima en el tiempo
                    # Desplazamiento mínimo (ignora coches aparcados que parpadean)
                    if t['frames'] >= MIN_FRAMES_VALIDO:
                        disp = math.hypot(t['actual'][0] - t['inicio'][0], t['actual'][1] - t['inicio'][1])
                        es_vehiculo = clase_id in CLASES_VEHICULOS
                        
                        if (not es_vehiculo and disp >= 20) or (es_vehiculo and disp >= MIN_DESPLAZAMIENTO):
                            # Retirar de otras clases si cambió la mayoría y añadir a la dominante
                            for c in conteo_unicos:
                                conteo_unicos[c].discard(track_id)
                            if clase_dominante in conteo_unicos:
                                conteo_unicos[clase_dominante].add(track_id)

                info_matricula = "No"
                caja_matricula = None

                # Detección y propagación de matrícula para vehículos
                if clase_id in CLASES_VEHICULOS:
                    vx1, vy1 = max(0, x1), max(0, y1)
                    vx2, vy2 = min(ancho, x2), min(alto, y2)
                    w_veh = vx2 - vx1
                    h_veh = vy2 - vy1

                    if w_veh > 45 and h_veh > 45:
                        usar_detector = True

                        if track_id != -1 and track_id in cache_matriculas:
                            rx1, ry1, rx2, ry2 = cache_matriculas[track_id]['rel_coords']
                            gx1 = int(vx1 + rx1 * w_veh)
                            gy1 = int(vy1 + ry1 * h_veh)
                            gx2 = int(vx1 + rx2 * w_veh)
                            gy2 = int(vy1 + ry2 * h_veh)
                            caja_matricula = (gx1, gy1, gx2, gy2)
                            info_matricula = f"[{gx1}:{gy1}:{gx2}:{gy2}]"
                            
                            if (frame_idx - cache_matriculas[track_id]['frame']) < 15:
                                usar_detector = False

                        if usar_detector:
                            roi_vehiculo = frame[vy1:vy2, vx1:vx2]
                            res_placas = modelo_matriculas(
                                roi_vehiculo, 
                                imgsz=320, 
                                conf=0.35, 
                                device=DISPOSITIVO, 
                                verbose=False
                            )

                            if res_placas and len(res_placas) > 0 and len(res_placas[0].boxes) > 0:
                                mejor_placa = res_placas[0].boxes[0]
                                px1_r, py1_r, px2_r, py2_r = map(int, mejor_placa.xyxy[0].tolist())
                                
                                gx1 = vx1 + px1_r
                                gy1 = vy1 + py1_r
                                gx2 = vx1 + px2_r
                                gy2 = vy1 + py2_r
                                caja_matricula = (gx1, gy1, gx2, gy2)
                                info_matricula = f"[{gx1}:{gy1}:{gx2}:{gy2}]"

                                if track_id != -1:
                                    cache_matriculas[track_id] = {
                                        'rel_coords': (px1_r / w_veh, py1_r / h_veh, px2_r / w_veh, py2_r / h_veh),
                                        'frame': frame_idx
                                    }

                # Escritura en CSV con el formato requerido por la práctica
                csv_writer.writerow([
                    frame_idx,
                    clase_dominante,
                    f"{confianza:.2f}",
                    track_id,
                    x1, y1, x2, y2,
                    info_matricula
                ])

                # Anotación gráfica del objeto
                color_obj = COLORES.get(clase_dominante, (0, 255, 0))
                cv2.rectangle(frame, (x1, y1), (x2, y2), color_obj, 2)
                cv2.putText(
                    frame, 
                    f"ID:{track_id} {clase_dominante} {confianza:.2f}", 
                    (x1, max(20, y1 - 8)), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_obj, 2
                )

                # Anotación gráfica de la matrícula
                if caja_matricula is not None:
                    mx1, my1, mx2, my2 = caja_matricula
                    cv2.rectangle(frame, (mx1, my1), (mx2, my2), COLORES['matricula'], 2)
                    cv2.putText(
                        frame, "Matricula", 
                        (mx1, max(15, my1 - 5)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, COLORES['matricula'], 2
                    )

        # Panel de conteo superior izquierdo
        cv2.rectangle(frame, (15, 15), (270, 160), (0, 0, 0), -1)
        cv2.rectangle(frame, (15, 15), (270, 160), (255, 255, 255), 1)
        cv2.putText(frame, f"Frame: {frame_idx}/{total_frames}", (25, 38), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, (255, 255, 255), 1)

        y_offset = 62
        for clase, ids in conteo_unicos.items():
            color_txt = COLORES.get(clase, (200, 200, 200))
            cv2.putText(frame, f"Total {clase}s: {len(ids)}", (25, y_offset), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_txt, 1)
            y_offset += 22

        out.write(frame)

        if frame_idx % 50 == 0 or frame_idx == total_frames:
            print(f"Progreso: {frame_idx}/{total_frames} fotogramas procesados...")

# CIERRE Y RESUMEN
cap.release()
out.release()
f_csv.close()

print()
print("¡Procesamiento finalizado!")
print(f"Vídeo generado: {VIDEO_SALIDA}")
print(f"CSV generado: {CSV_SALIDA}")
print("Totales detectados:")
for clase, ids in conteo_unicos.items():
    print(f" - {clase}: {len(ids)}")


Dispositivo de ejecución asignado: cpu
Procesando 2832 fotogramas...
Progreso: 50/2832 fotogramas procesados...
Progreso: 100/2832 fotogramas procesados...
Progreso: 150/2832 fotogramas procesados...
Progreso: 200/2832 fotogramas procesados...
Progreso: 250/2832 fotogramas procesados...
Progreso: 300/2832 fotogramas procesados...
Progreso: 350/2832 fotogramas procesados...
Progreso: 400/2832 fotogramas procesados...
Progreso: 450/2832 fotogramas procesados...
Progreso: 500/2832 fotogramas procesados...
Progreso: 550/2832 fotogramas procesados...
Progreso: 600/2832 fotogramas procesados...
Progreso: 650/2832 fotogramas procesados...
Progreso: 700/2832 fotogramas procesados...
Progreso: 750/2832 fotogramas procesados...
Progreso: 800/2832 fotogramas procesados...
Progreso: 850/2832 fotogramas procesados...
Progreso: 900/2832 fotogramas procesados...
Progreso: 950/2832 fotogramas procesados...
Progreso: 1000/2832 fotogramas procesados...
Progreso: 1050/2832 fotogramas procesados...
Progre